## 1. Carga de datos
Cargamos el fichero CSV con las columnas más relevantes

In [3]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("consumptions.csv", low_memory=False)

columns_needed = [
"userId",
"city",
"country",
"gender",
"yearOfBirth",
"homeEnergyLabel",
"householdProfile",
"category",
"value",
"carbonEmissions",
"energyExpended",
"transportation.transportationType",
"transportation.publicVehicleOccupancy",
"transportation.privateVehicleOccupancy",
"electricity.householdSize",
"heating.householdSize",
"heating.heatingFuel",
"electricity.electricitySource"
]

df = df[columns_needed]

display(df.head())
display(df.shape)

,userId,city,country,gender,yearOfBirth,homeEnergyLabel,householdProfile,category,value,carbonEmissions,energyExpended,transportation.transportationType,transportation.publicVehicleOccupancy,transportation.privateVehicleOccupancy,electricity.householdSize,heating.householdSize,heating.heatingFuel,electricity.electricitySource
0,c48be05f22d807690af96730960b5741b9e0cc2b66afec...,Madrid,ES,female,2005.0,NaN,NaN,transportation,7.0,0.0952,0.4991,metroTramOrUrbanLightTrain,medium,NaN,NaN,NaN,NaN,NaN
1,c48be05f22d807690af96730960b5741b9e0cc2b66afec...,Madrid,ES,female,2005.0,NaN,NaN,transportation,7.0,0.0952,0.4991,metroTramOrUrbanLightTrain,medium,NaN,NaN,NaN,NaN,NaN
2,c48be05f22d807690af96730960b5741b9e0cc2b66afec...,Madrid,ES,female,2005.0,NaN,NaN,transportation,7.0,0.0952,0.4991,metroTramOrUrbanLightTrain,medium,NaN,NaN,NaN,NaN,NaN
3,c48be05f22d807690af96730960b5741b9e0cc2b66afec...,Madrid,ES,female,2005.0,NaN,NaN,transportation,7.0,0.0952,0.4991,metroTramOrUrbanLightTrain,medium,NaN,NaN,NaN,NaN,NaN
4,c48be05f22d807690af96730960b5741b9e0cc2b66afec...,Madrid,ES,female,2005.0,NaN,NaN,transportation,7.0,0.0952,0.4991,metroTramOrUrbanLightTrain,medium,NaN,NaN,NaN,NaN,NaN


(119124, 18)

## 2. Filtrado y Limpieza de Datos

### 2.1. Tratamiento de la variable 'yearOfBirth' y segmentación generacional

Implementación de un filtro para la detección de valores anómalos (*outliers*) en el año de nacimiento, asignándolos como nulos (NaN) aquellos registros inconsistentes. Adicionalmente, se genera una variable derivada categórica para clasificar a los usuarios según su cohorte generacional.

In [4]:
# Generación de una copia de trabajo del DataFrame
df_filtered = df.copy()

low_age = (df_filtered['yearOfBirth'] < 1950).sum()
high_age = (df_filtered['yearOfBirth'] > 2007).sum()

print(f"Se han vuelto nulos {low_age} datos menores a 1950 y {high_age} valores mayores a 2007. Total: {low_age + high_age}")

# Usamos una máscara para asignar None (que se verá como NaN en el dataframe)
mask = (df_filtered['yearOfBirth'] < 1950) | (df_filtered['yearOfBirth'] > 2007)
df_filtered.loc[mask, 'yearOfBirth'] = None

# Crear grupos de edad
import numpy as np

conditions = [
    (df_filtered['yearOfBirth'] >= 1949) & (df_filtered['yearOfBirth'] <= 1968),
    (df_filtered['yearOfBirth'] >= 1969) & (df_filtered['yearOfBirth'] <= 1980),
    (df_filtered['yearOfBirth'] >= 1981) & (df_filtered['yearOfBirth'] <= 1993),
    (df_filtered['yearOfBirth'] >= 1994) & (df_filtered['yearOfBirth'] <= 2010),
    (df_filtered['yearOfBirth'] >= 2011) & (df_filtered['yearOfBirth'] <= 2024),
]

choices = [
    "BabyBoom",
    "GenX",
    "GenY",
    "GenZ",
    "Alfa"
]

df_filtered['generation'] = np.select(conditions, choices, default="Unknown")

Se han vuelto nulos 85 datos menores a 1950 y 14381 valores mayores a 2007. Total: 14466


## 3. Segmentación por Categorías y Limpieza de Consumos Incongruentes

Separación del conjunto de datos en tres subgrupos independientes según el tipo de consumo energético (`electricity`, `transportation` y `heating`). Para cada categoría, se aplican umbrales específicos de validez técnica y lógica con el fin de detectar y depurar valores atípicos (*outliers*), asignándolos como nulos (`NaN`):

* **Electricidad:** Exclusión de registros fuera del rango [50, 5000) kWh.
* **Transporte:** Eliminación de consumos negativos, valores superiores a 12432 kWh, y registros con consumo igual a cero (exceptuando aquellos trayectos realizados a pie o en bicicleta, los cuales se consideran lógicamente válidos).
* **Calefacción:** Filtrado y remoción de registros con consumos inferiores a 50 kWh.

In [5]:
import numpy as np

# Filtros iniciales de electricidad, transporte y calefacción (copias explícitas)
df_elec = df_filtered[df_filtered['category'] == 'electricity'].copy()
df_transport = df_filtered[df_filtered['category'] == 'transportation'].copy()
df_heating = df_filtered[df_filtered['category'] == 'heating'].copy()

# --- ELECTRICITY ---
# 1. Contar outliers
low_elec = (df_elec['energyExpended'] < 50).sum()
high_elec = (df_elec['energyExpended'] > 5000).sum()

print(f"Se han vuelto nulos {low_elec} datos menores a 50 y {high_elec} valores mayores a 5000 en la categoría electricity. Total: {low_elec + high_elec}")

# 2. Convertir a NaN los que cumplen las condiciones de eliminación
condicion_eliminar_elec = (df_elec['energyExpended'] < 50) | (df_elec['energyExpended'] >= 5000)
df_elec.loc[condicion_eliminar_elec, 'energyExpended'] = np.nan


# --- TRANSPORT ---
# 1. Contar outliers
low_trans_strict = (df_transport['energyExpended'] < 0).sum()
zero_trans_to_remove = (
    (df_transport['energyExpended'] == 0) & 
    (~df_transport['transportation.transportationType'].isin(['walking', 'bike']))
).sum()

low_trans = low_trans_strict + zero_trans_to_remove
high_trans = (df_transport['energyExpended'] > 12432).sum()

print(f"Se han vuelto nulos {low_trans} datos menores o iguales a 0 (según reglas) y {high_trans} valores mayores a 12432 en la categoría transport. Total: {low_trans + high_trans}")

# 2. Convertir a NaN los que NO cumplen tus condiciones de conservación
condicion_valores_normales = (df_transport['energyExpended'] > 0) & (df_transport['energyExpended'] <= 12432)
condicion_ceros_validos = (df_transport['energyExpended'] == 0) & (df_transport['transportation.transportationType'].isin(['walking', 'bike']))

# Negamos la condición de guardado (~): todo lo que NO sea válido, pasa a ser NaN
df_transport.loc[~(condicion_valores_normales | condicion_ceros_validos), 'energyExpended'] = np.nan


# --- HEATING ---
# 1. Contar outliers (Nota: en tu print original decías '< 0' pero luego filtrabas '>= 50'. He ajustado el conteo a tu filtro real)
low_heat = (df_heating['energyExpended'] < 50).sum()

print(f"Se han vuelto nulos {low_heat} datos menores a 50 en la categoría heating")

# 2. Convertir a NaN los menores a 50
df_heating.loc[df_heating['energyExpended'] < 50, 'energyExpended'] = np.nan

Se han vuelto nulos 231 datos menores a 50 y 15 valores mayores a 5000 en la categoría electricity. Total: 246
Se han vuelto nulos 358 datos menores o iguales a 0 (según reglas) y 4 valores mayores a 12432 en la categoría transport. Total: 362
Se han vuelto nulos 128 datos menores a 50 en la categoría heating


## 4. Consolidación del Dataset Global y Exportación por Categorías

En esta fase final, se integra la limpieza realizada en los subgrupos de vuelta en el conjunto de datos maestro mediante el método `.update()`. Posteriormente, se realiza una depuración de columnas (*feature selection*) para eliminar las variables irrelevantes de las otras categorías en cada fichero específico, procediendo a la exportación de los cuatro archivos definitivos en formato CSV:

* **`clean_dataset.csv`**: El dataset global unificado con todos los consumos depurados.
* **`electricity.csv`, `transport.csv` y `heating.csv`**: Los tres datasets específicos por área, optimizados sin columnas huérfanas o redundantes, listos para su explotación estadística en SPSS.

In [6]:
df_filtered.update(df_elec)
df_filtered.update(df_transport)
df_filtered.update(df_heating)

# Exportar dataset limpio
df_filtered.to_csv("clean_dataset.csv", index=False)

# --- ELECTRICITY ---
df_elec = df_elec.loc[:, ~df_elec.columns.str.startswith(('transportation.', 'heating.'))]

# --- TRANSPORT ---
df_transport = df_transport.loc[:, ~df_transport.columns.str.startswith(('electricity.', 'heating.'))]

# --- HEATING ---
df_heating = df_heating.loc[:, ~df_heating.columns.str.startswith(('electricity.', 'transportation.'))]

# Opcional: por categorías
df_elec.to_csv("electricity.csv", index=False)
df_transport.to_csv("transport.csv", index=False)
df_heating.to_csv("heating.csv", index=False)

# Mensajes de confirmación para verificar la correcta exportación
print("¡Procesamiento y exportación completados con éxito!")
print(f" -> Dataset global exportado: {df_filtered.shape[0]} filas y {df_filtered.shape[1]} columnas.")
print(f" -> Dataset de Electricidad optimizado: {df_elec.shape[0]} filas y {df_elec.shape[1]} columnas.")
print(f" -> Dataset de Transporte optimizado: {df_transport.shape[0]} filas y {df_transport.shape[1]} columnas.")
print(f" -> Dataset de Calefacción optimizado: {df_heating.shape[0]} filas y {df_heating.shape[1]} columnas.")

¡Procesamiento y exportación completados con éxito!
 -> Dataset global exportado: 119124 filas y 19 columnas.
 -> Dataset de Electricidad optimizado: 964 filas y 14 columnas.
 -> Dataset de Transporte optimizado: 117712 filas y 15 columnas.
 -> Dataset de Calefacción optimizado: 448 filas y 14 columnas.
